In [4]:
# imports
from io import StringIO
import pandas as pd
import requests
import yfinance as yf


In [5]:

# define headers to mimic a real browser and avoid HTTP 403 errors
headers = {
    "User-Agent": "Mozilla/5.0"
}

# URL of Wikipedia page containing the list of S&P 500 companies
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# send HTTP GET request with headers to retrieve the HTML content
response = requests.get(url, headers=headers)
html = response.text

# parse all HTML tables from the page into a list of df
tables = pd.read_html(StringIO(html))

# first table contains S&P 500 companies
sp500_table = tables[0]


In [6]:
# extract ticker symbols
tickers = (
    sp500_table["Symbol"]
    .str.replace(".", "-", regex=False)
    .tolist()
)

# print lenght
len(tickers)


503

In [7]:
# save ticker list locally for reproducibility
sp500_table.to_csv("sp500_constituents.csv", index=False)


In [8]:
# load S&P 500 constituents from local CSV
sp500_table = pd.read_csv("sp500_constituents.csv")

In [9]:
# show first 5 rows
sp500_table.head()


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [10]:
# print first 10 tickers
tickers[:10]


['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']

In [11]:
# download daily adjusted close prices for last 5 years
raw_data = yf.download(
    tickers=tickers,
    period="5y",
    interval="1d",
    auto_adjust=True,
    threads=True
)


[*********************100%***********************]  503 of 503 completed


In [12]:
# inspect structure
raw_data.head()


Price            Close                                                \
Ticker               A        AAPL       ABBV        ABNB        ABT   
Date                                                                   
2020-12-14  112.840088  118.547028  85.744019  130.000000  97.567673   
2020-12-15  113.468376  124.485077  84.608444  124.800003  98.170670   
2020-12-16  113.323380  124.416931  86.106102  137.990005  98.070168   
2020-12-17  114.995560  125.283325  86.311813  147.050003  99.385826   
2020-12-18  115.314568  123.297470  85.949738  157.300003  99.559402   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2020-12-14  31.921663  226.880035  486.420013  130.427338  42.357525  ...   
2020-12-15  33.214886  229.951675  482.640015  132.012665  43.123062  ...   
2020-12-16  33.661808  230.324020  489.899994  130.858017  42.801189  ...   
2020-12-17  33.804443  246.166000  495.359985  131.756027  42.957783  ...   
2020-12-18  32.739437  247.822830  502.950012  132.470825  43.218758  ...   

Price        Volume                                                          \
Ticker           WY     WYNN      XEL       XOM      XYL       XYZ      YUM   
Date                                                                          
2020-12-14  4789900  2856100  2355600  30595000   682400   7379000  2015300   
2020-12-15  5407900  3196300  1977400  27138400   881900   5713600  2357700   
2020-12-16  4375700  2319500  1991600  34273000   891800   8216500  2412400   
2020-12-17  5326300  2046300  2959800  21204600   964800  10440100  2842700   
2020-12-18  7757000  3196300  5993500  46596800  1774600   8099500  4513800   

Price                                 
Ticker          ZBH    ZBRA      ZTS  
Date                                  
2020-12-14   995598  309100  1612900  
2020-12-15  1004044  311700  1813100  
2020-12-16   765393  474300  1440900  
2020-12-17   828738  819500  1486800  
2020-12-18  1819598  900400  3566400  

[5 rows x 2515 columns]

In [13]:
# extract closing prices
close_prices = raw_data["Close"]

# first rows
close_prices.head()

# shape of price matrix
close_prices.shape


(1256, 503)

In [14]:
# save raw price data
close_prices.to_csv("sp500_close_prices_raw.csv")


In [15]:
# looking at missing values per stock
close_prices.isna().sum().sort_values(ascending=False).head(10)


Ticker
Q       1222
SOLS    1217
SNDK    1046
GEV      825
SOLV     824
VLTO     705
KVUE     600
GEHC     505
CEG      276
HOOD     156
dtype: int64

In [16]:
# cleaning df

# calculate minimum required days (95% coverage)
total_days = len(close_prices)
min_required_days = int(total_days * 0.95)

print(f"Total trading days: {total_days}")
print(f"Minimum required: {min_required_days} days (95% coverage)")

# keep stocks with sufficient data
data_coverage = close_prices.notna().sum()
valid_stocks = data_coverage[data_coverage >= min_required_days].index

print(f"\nStocks with sufficient data: {len(valid_stocks)} / {len(close_prices.columns)}")
print(f"Stocks dropped: {len(close_prices.columns) - len(valid_stocks)}")

# filter dataset
clean_prices = close_prices[valid_stocks]
print(f"\n Clean dataset shape: {clean_prices.shape}")


Total trading days: 1256
Minimum required: 1193 days (95% coverage)

Stocks with sufficient data: 491 / 503
Stocks dropped: 12

 Clean dataset shape: (1256, 491)


In [17]:
#missing_per_day = clean_prices.isna().sum(axis=1)
#missing_per_day.describe()


In [18]:
clean_prices_ffill = clean_prices.ffill().dropna()
clean_prices_ffill.isna().sum().sum()
clean_prices_ffill.shape


(1217, 491)

In [19]:
clean_prices_ffill.to_csv("sp500_close_prices_clean.csv")

In [20]:
print("=== Final Cleaned Dataset ===\n")

print(f"Number of trading days: {clean_prices_ffill.shape[0]}")
print(f"Number of stocks: {clean_prices_ffill.shape[1]}")
print(f"Date range: {clean_prices_ffill.index[0]} to {clean_prices_ffill.index[-1]}")

# first few rows
print("\nFirst 5 rows of cleaned prices:")
print(clean_prices_ffill.head())

=== Final Cleaned Dataset ===

Number of trading days: 1217
Number of stocks: 491
Date range: 2021-02-10 00:00:00 to 2025-12-12 00:00:00

First 5 rows of cleaned prices:
Ticker               A        AAPL       ABBV        ABNB         ABT  \
Date                                                                    
2021-02-10  121.099747  131.992661  86.621986  211.660004  114.931175   
2021-02-11  122.919930  131.739166  86.372253  216.839996  116.353020   
2021-02-12  123.907478  131.973175  86.946678  212.679993  117.628082   
2021-02-16  123.878410  129.847885  86.746880  209.860001  117.435471   
2021-02-17  125.979347  127.556816  88.486809  201.960007  116.719933   

Ticker           ACGL         ACN        ADBE         ADI        ADM  ...  \
Date                                                                  ...   
2021-02-10  33.176849  239.974411  492.670013  140.415771  47.708275  ...   
2021-02-11  33.813953  241.870377  496.619995  145.520004  48.111237  ...   
2021-02-12